# Clase 04: CNNs y reconocimiento visual

Las CNNs existen porque las imagenes tienen estructura espacial local. Una red totalmente conectada ignora esa estructura y aprende muchos parametros redundantes. Convolucionar es imponer un sesgo util: patrones pequenos que se reutilizan sobre toda la imagen.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset


import tools
from tools.notebook_utils import choose_value, configure_runtime, count_parameters, plot_history

runtime = configure_runtime(seed=23)
print(runtime.summary())
        


## Idea clave

Una CNN trabaja con tres intuiciones que siguen vigentes en 2026:

- vecindad local: un borde o una textura se detectan mejor con filtros pequenos,
- comparticion de pesos: el mismo patron puede aparecer en cualquier region,
- jerarquia: capas tempranas capturan rasgos simples y capas profundas capturan composiciones mas complejas.

Los Vision Transformers importan mucho en modelos grandes, pero en entornos pequenos, edge o datasets modestos las CNNs siguen siendo una base excelente.


In [ ]:
digits = load_digits()
images = (digits.images / 16.0).astype('float32')
labels = digits.target.astype('int64')

X_train, X_temp, y_train, y_temp = train_test_split(images, labels, test_size=0.3, random_state=23, stratify=labels)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=23, stratify=y_temp)

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for axis, image, label in zip(axes.ravel(), X_train[:10], y_train[:10]):
    axis.imshow(image, cmap='gray')
    axis.set_title(f'etiqueta={label}')
    axis.axis('off')
plt.tight_layout()
        


In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, augment: bool = False) -> None:
        self.images = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image = self.images[index].clone()
        if self.augment:
            shift_y = int(torch.randint(-1, 2, (1,)).item())
            shift_x = int(torch.randint(-1, 2, (1,)).item())
            image = torch.roll(image, shifts=(shift_y, shift_x), dims=(1, 2))
            image = image + 0.03 * torch.randn_like(image)
            image = image.clamp(0.0, 1.0)
        return image, self.labels[index]


train_loader = DataLoader(DigitsDataset(X_train, y_train, augment=True), batch_size=int(choose_value(64, 32)), shuffle=True)
val_loader = DataLoader(DigitsDataset(X_val, y_val), batch_size=128)
test_loader = DataLoader(DigitsDataset(X_test, y_test), batch_size=128)


class MLPDigits(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(64, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 10),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.network(inputs)


class SimpleCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2),
            torch.nn.Conv2d(16, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = torch.nn.Linear(32, 10)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x = self.features(inputs)
        x = x.flatten(start_dim=1)
        return self.classifier(x)


print({'mlp_params': count_parameters(MLPDigits()), 'cnn_params': count_parameters(SimpleCNN())})
        


In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
epochs = int(choose_value(20, 8))


def train_image_model(model: torch.nn.Module) -> tuple[dict[str, list[float]], float]:
    model = model.to(runtime.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-3)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        train_loss_total = 0.0
        for images_batch, labels_batch in train_loader:
            images_batch = images_batch.to(runtime.device)
            labels_batch = labels_batch.to(runtime.device)
            optimizer.zero_grad()
            logits = model(images_batch)
            loss = loss_fn(logits, labels_batch)
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item() * len(images_batch)

        model.eval()
        val_loss_total = 0.0
        predictions = []
        targets = []
        with torch.inference_mode():
            for images_batch, labels_batch in val_loader:
                images_batch = images_batch.to(runtime.device)
                labels_batch = labels_batch.to(runtime.device)
                logits = model(images_batch)
                val_loss_total += loss_fn(logits, labels_batch).item() * len(images_batch)
                predictions.extend(logits.argmax(dim=1).cpu().tolist())
                targets.extend(labels_batch.cpu().tolist())

        history['train_loss'].append(train_loss_total / len(train_loader.dataset))
        history['val_loss'].append(val_loss_total / len(val_loader.dataset))
        history['val_acc'].append(accuracy_score(targets, predictions))

    model.eval()
    test_predictions = []
    test_targets = []
    with torch.inference_mode():
        for images_batch, labels_batch in test_loader:
            images_batch = images_batch.to(runtime.device)
            logits = model(images_batch)
            test_predictions.extend(logits.argmax(dim=1).cpu().tolist())
            test_targets.extend(labels_batch.tolist())

    return history, accuracy_score(test_targets, test_predictions)


mlp_history, mlp_test_acc = train_image_model(MLPDigits())
cnn_history, cnn_test_acc = train_image_model(SimpleCNN())
print({'mlp_test_acc': round(mlp_test_acc, 3), 'cnn_test_acc': round(cnn_test_acc, 3)})
        


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(mlp_history['val_acc'], label='MLP')
plt.plot(cnn_history['val_acc'], label='CNN')
plt.title('Comparacion de accuracy en validacion')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()

sample_image = torch.tensor(X_test[0], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
edge_filter = torch.tensor([[[[-1.0, -1.0, -1.0], [-1.0, 8.0, -1.0], [-1.0, -1.0, -1.0]]]])
feature_map = torch.nn.functional.conv2d(sample_image, edge_filter, padding=1).squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(sample_image.squeeze().numpy(), cmap='gray')
axes[0].set_title('Imagen original')
axes[0].axis('off')
axes[1].imshow(feature_map, cmap='inferno')
axes[1].set_title('Filtro 3x3 sobre la imagen')
axes[1].axis('off')
plt.tight_layout()
        


In [ ]:
if runtime.online_mode:
    print('Modo online activo: intentando cargar FashionMNIST como apendice opcional.')
    fashion_train = torchvision.datasets.FashionMNIST(root=str(tools.ROOT_DIR / '.artifacts' / 'fashion_mnist'), train=True, download=True)
    print({'fashion_mnist_examples': len(fashion_train)})
else:
    print('HENRY_DL_ONLINE_MODE=0 -> el apendice online queda desactivado.')
        


In [ ]:
# Mapas de caracteristicas aprendidos por la CNN
import matplotlib.pyplot as plt
import torch
import numpy as np

# Reutilizar el CNN ya entrenado
cnn_model = SimpleCNN().to(runtime.device)
# Re-entrenar brevemente para obtener pesos representativos
_cnn_hist, _ = train_image_model(cnn_model)
cnn_model.eval()

# Tomar 4 ejemplos del test set
test_images_sample = torch.tensor(X_test[:4], dtype=torch.float32).unsqueeze(1).to(runtime.device)
test_labels_sample = y_test[:4]

# Extraer salidas intermedias via hooks
conv1_maps = []
conv2_maps = []

def hook_conv1(module, input, output):
    conv1_maps.append(output.detach().cpu())

def hook_conv2(module, input, output):
    conv2_maps.append(output.detach().cpu())

# features[0] = Conv2d(1,16), features[1] = ReLU, features[2] = MaxPool2d
# features[3] = Conv2d(16,32), features[4] = ReLU
h1 = cnn_model.features[0].register_forward_hook(hook_conv1)
h2 = cnn_model.features[3].register_forward_hook(hook_conv2)

with torch.inference_mode():
    _ = cnn_model(test_images_sample)
h1.remove()
h2.remove()

conv1_out = conv1_maps[0]  # (4, 16, 8, 8)
conv2_out = conv2_maps[0]  # (4, 32, 4, 4)

# Mostrar Conv1 (16 canales) para la primera imagen: grid 4x4
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ch_idx, ax in enumerate(axes.ravel()):
    fmap = conv1_out[0, ch_idx].numpy()
    ax.imshow(fmap, cmap='viridis', interpolation='nearest')
    ax.set_title(f'Canal {ch_idx}', fontsize=7)
    ax.axis('off')
plt.suptitle(
    f'Conv1 (16 canales, etiqueta={test_labels_sample[0]})\n'
    'Detecta bordes y texturas simples',
    fontsize=11
)
plt.tight_layout()

# Mostrar Conv2 (32 canales) para la primera imagen: grid 4x8
fig2, axes2 = plt.subplots(4, 8, figsize=(14, 7))
for ch_idx, ax in enumerate(axes2.ravel()):
    fmap = conv2_out[0, ch_idx].numpy()
    ax.imshow(fmap, cmap='plasma', interpolation='nearest')
    ax.set_title(f'C{ch_idx}', fontsize=6)
    ax.axis('off')
plt.suptitle(
    f'Conv2 (32 canales, etiqueta={test_labels_sample[0]})\n'
    'Detecta patrones mas complejos (combinaciones de bordes)',
    fontsize=11
)
plt.tight_layout()
# Observar: Conv1 produce mapas que recuerdan filtros de borde/textura
# Conv2 combina esas respuestas en patrones de mayor nivel: jerarquia de caracteristicas


## Para cerrar

### Que deberias poder responder

- por que compartir pesos reduce parametros sin perder capacidad local,
- por que una CNN suele ganar frente a una MLP en datos visuales pequenos,
- por que `pooling` y receptive field ayudan a abstraer patron espacial.

### Ideas para experimentar

- reemplaza `MaxPool2d` por `AvgPool2d`,
- quita augmentation y compara el test,
- aumenta la profundidad de la CNN y mide si el beneficio justifica el costo.
